In [ ]:
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
pio.templates.default = "plotly_white"
# Warm up kaleido so the first write_image call doesn't block long enough to trigger an IOPub timeout
go.Figure().write_image("/tmp/_kaleido_warmup.png")
import xarray as xr
from whakaaribn.visualize import trellis_plot
from whakaaribn import get_color

In [ ]:
try:
    data_file = snakemake.input.data
    forecast_all_data = snakemake.input.forecast_all_data
    forecast_uncertainty = snakemake.input.forecast_uncertainty
except NameError as e:
    data_file = '../data/whakaari_data_with_groups.csv'
    forecast_all_data = '../forecasts/whakaari_forecasts.nc'
    forecast_uncertainty = '../forecasts/whakaari_uncertainty.nc'

In [ ]:
xds_best = xr.open_dataset(forecast_all_data)
xds_all_1 = xr.open_dataset(forecast_uncertainty)
data = pd.read_csv(data_file, parse_dates=True, index_col=0)

In [ ]:
median_model = xds_all_1.median('model_score').to_array()
models = {"Eruption Probability (median model)": {'model': median_model.squeeze('variable'), 'color': get_color(0)},
          "Eruption Probability (best model)": {'model': xds_best['probs'], 'color': get_color(1)},
          "ensemble": {'model': xds_all_1.to_array().squeeze('variable'), 'colorscale': "algae"},
}
fig = trellis_plot(models, data, plot_uncertainty='ensemble')
try:
    fig.write_image(snakemake.output.forecast_ensemble_plot, width=1200, height=1000, scale=5)
except NameError:
    pass